In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import os
import seaborn as sns
import pandas as pd
from collections import Counter
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA

## Caricamento dei files

In [ ]:
# Imposta il percorso della cartella contenente i file pkl
path = '/content/Dataset'
arrays = []
data_list = []

# Itera su tutti i file pkl nella cartella e caricali in una lista
for filename in os.listdir(path):
    if filename.endswith('.pkl'):
        with open(os.path.join(path, filename), 'rb') as f:
            # Carica l'array NumPy dal file e seleziona solo il secondo canale
            arr = np.asarray(pickle.load(f))[:, 1:2, :]
            arrays.append(arr)
            data_list.append(os.path.splitext(filename)[0])

print(data_list)

In [ ]:
all_data = arrays[5]
#arrays = [np.array(arrays[6]), np.array(arrays[4]), np.array(arrays[2]), np.array(arrays[1]), np.array(arrays[5]), np.array(arrays[0])]

## Bilanciamento del dataset

In [ ]:
# Calcola la dimensione minima tra gli array nella lista
#min_len = min([arr.shape[0] for arr in arrays])
min_len = 2000

# Bilancia gli array nella lista
balanced_arrays = [arr for arr in arrays]

# Unisci gli array bilanciati in un unico array NumPy
data = np.concatenate(balanced_arrays, axis=0)

data = np.squeeze(data,1)
data.shape

In [ ]:
def calculate_features(window):
    # Calcola la media del segnale nella finestra
    mean = np.mean(window)

    # Calcola la deviazione standard del segnale nella finestra
    std_dev = np.std(window)

    # Calcola il massimo valore nella finestra
    max_val = np.max(window)

    # Calcola il minimo valore nella finestra
    min_val = np.min(window)

    # Calcola la differenza tra il massimo e il minimo
    peak_to_peak = max_val - min_val

    return [mean, std_dev, max_val, min_val, peak_to_peak]

In [ ]:
def normalize_data(data):
    # Calcola la media e la deviazione standard lungo ciascuna feature (colonna)
    mean = np.mean(data, axis=0)
    std_dev = np.std(data, axis=0)

    # Applica la standardizzazione (z-score normalization)
    normalized_data = (data - mean) / std_dev

    return normalized_data

In [ ]:
pca = PCA(n_components=2)

def preprocessing(data):
    # Lunghezza della finestra e sovrapposizione
    #window_size = 30
    #overlap = 2

    # Calcola tutte le finestre scorrevoli
    #windows = []
    #for signal in data:
    #    n = len(signal)
    #    indices = np.arange(0, n - window_size + 1, overlap)
    #    windows.extend(signal[i:i+window_size] for i in indices)

    # Calcola le features per tutte le finestre
    #features = np.array([calculate_features(window) for window in windows])
    features = np.array([calculate_features(signal) for signal in data])

    # Normalizza i dati
    #normalized_data = normalize_data(features)

    # Riduzione della dimensionalità
    #dataset = pca.fit_transform(dataset)

    # Mischia casualmente i dati
    np.random.shuffle(features)

    return features


## Training DBSCAN

In [ ]:
dataset = preprocessing(data)[:10000000]
# crea il modello di clustering con DBSCAN
kmeans = KMeans(n_clusters=3, random_state=42)  # Imposta il numero di cluster desiderato
kmeans.fit(dataset)
clustering_labels = kmeans.labels_

In [ ]:
import time
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
import matplotlib.pyplot as plt
from tqdm import tqdm

def measure_training_time_kmeans(dataset_size):
    dataset = preprocessing(data)[:dataset_size]

    start_time = time.time()
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    kmeans.fit(dataset)
    end_time = time.time()

    return end_time - start_time

def measure_training_time_dbscan(dataset_size):
    dataset = preprocessing(data)[:dataset_size]

    start_time = time.time()
    dbscan = DBSCAN()
    dbscan.fit(dataset)
    end_time = time.time()

    return end_time - start_time

# Define the dataset sizes to test
dataset_sizes = range(10000, 100000, 10000)

# Measure training time for each dataset size with KMeans
training_times_kmeans = []
for size in tqdm(dataset_sizes, desc='Progress', unit='dataset'):
    elapsed_time = measure_training_time_kmeans(size)
    training_times_kmeans.append(elapsed_time)

# Measure training time for each dataset size with DBSCAN
training_times_dbscan = []
for size in tqdm(dataset_sizes, desc='Progress', unit='dataset'):
    elapsed_time = measure_training_time_dbscan(size)
    training_times_dbscan.append(elapsed_time)

# Create the line plot for KMeans
plt.plot(dataset_sizes, training_times_kmeans, marker='o', label='KMeans')

# Create the line plot for DBSCAN
plt.plot(dataset_sizes, training_times_dbscan, marker='o', label='DBSCAN')

# Add legend
plt.legend()

# Set plot title and labels
plt.title('Training Time vs. Dataset Size')
plt.xlabel('Dataset Size')
plt.ylabel('Training Time (s)')

# Show the plot
plt.show()

In [ ]:
!cat /proc/cpuinfo


In [ ]:
# Visualizza i risultati del clustering
n_clusters = len(set(clustering_labels)) - (1 if -1 in clustering_labels else 0)
print("Numero di cluster trovati:", n_clusters)

# Visualizza i punti anomali (cluster con etichetta -1)
n_anomalies = list(clustering_labels).count(-1)
print("Numero di punti anomali trovati:", n_anomalies)

In [ ]:
labelM = ["Normal","Maintenance","Compressor Fault"]
# Creare un DataFrame per i dati e le etichette di clustering
df = pd.DataFrame({'X': dataset[:, 0], 'Y': dataset[:, 1], 'Cluster': clustering_labels})

# Plottiamo i cluster usando Seaborn
sns.scatterplot(data=df, x='X', y='Y', hue='Cluster', palette='viridis', alpha=0.5)
plt.show()

In [ ]:
label_names = ["Maintenance", "Compressor Fault","Normal" ]

# Creare un DataFrame per i dati e le etichette di clustering
df = pd.DataFrame({'Mean': dataset[:, 0], 'Std': dataset[:, 1], 'Cluster': clustering_labels})

# Convertire le etichette dei cluster in nomi corrispondenti
df['Cluster'] = df['Cluster'].map({i: name for i, name in enumerate(label_names)})

# Plottare i cluster usando Seaborn
sns.scatterplot(data=df, x='Mean', y='Std', hue='Cluster', palette='viridis', alpha=0.8)

# Personalizzare la leggenda
plt.legend(title='Cluster')
plt.show()

#Inference

In [ ]:
# nuovo_campione = il tuo nuovo campione
# data = i tuoi dati di addestramento
# clustering_labels = le etichette di clustering assegnate ai dati di addestramento

def distance(nuovo_campione, data, clustering_labels, metric='chebyshev'):
    # Calcola il centroide di ciascun cluster
    cluster_centroids = []
    for label in np.unique(clustering_labels):
        cluster_points = data[clustering_labels == label]
        centroid = np.mean(cluster_points, axis=0)
        cluster_centroids.append(centroid)

    cluster_centroids = np.array(cluster_centroids)

    # Calcola la distanza tra il nuovo campione e i centroidi dei cluster
    distanze = pairwise_distances(nuovo_campione.reshape(1, -1), cluster_centroids, metric=metric)

    # Identifica il cluster più vicino
    cluster_più_vicino = np.argmin(distanze)
    return cluster_più_vicino

In [ ]:
# Calcola la dimensione minima tra gli array nella lista
#min_len = min([arr.shape[0] for arr in arrays])
min_len = 2000

inference_data = all_data

inference_data = np.squeeze(inference_data,1)
inference_data.shape

In [ ]:
inference_data = preprocessing(inference_data)

In [ ]:
# Creare un DataFrame per i dati e le etichette di clustering
df2 = pd.DataFrame({'Mean': inference_data[:, 0], 'Std': inference_data[:, 4]})

# Plottiamo i cluster usando Seaborn
sns.scatterplot(data=df2, x='Mean', y='Std', palette='viridis', alpha=0.5)
sns.scatterplot(data=df, x='Mean', y='Std', hue='Cluster', palette='viridis', alpha=0.5)
plt.show()

In [ ]:
result = []
for elem in inference_data:
  result.append(distance(elem, dataset, clustering_labels))
count = Counter(result)
elemento_piu_ripetuto = count.most_common(1)[0][0]
print(f"Cluster più vicino al nuovo campione: {elemento_piu_ripetuto}")

In [ ]:
# Salva il modello con pickle
with open('kmeans_model.pkl', 'wb') as model_file:
    pickle.dump(kmeans, model_file)

In [ ]:
prova = inference_data[35]
prova = np.expand_dims(prova,0)
new_labels = kmeans.predict(prova)
print(new_labels)

In [ ]:
def compute_distances_from_clusters(kmeans_model, new_sample):
    """
    Calcola la distanza del nuovo campione dai centroidi dei cluster.

    Parametri:
    - kmeans_model: Modello KMeans già addestrato
    - new_sample: Nuovo campione da valutare

    Restituisce:
    - distances: Array delle distanze tra il nuovo campione e i centroidi dei cluster
    """


    # Calcola la distanza dai centroidi dei cluster
    distances = kmeans_model.transform(new_sample)

    return distances

In [ ]:
# Assumendo che il tuo dataset sia memorizzato nella variabile 'dataset'
# Calcola le distanze
distances = compute_distances_from_clusters(kmeans, dataset)

# Plotta la distribuzione delle distanze
plt.figure(figsize=(10, 6))
plt.hist(np.min(distances, axis=1), bins=30, edgecolor='black')
plt.title('Distribution of Distances from Cluster Centroids')
plt.xlabel('Minimum Distance')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Ottieni i centroidi dei cluster
centroids = kmeans.cluster_centers_

# Calcola la distanza tra i centroidi
distances_between_centroids = np.linalg.norm(centroids[0] - centroids[1])

print("Distanza tra i centroidi:", distances_between_centroids)

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

silhouette_avg = silhouette_score(dataset, clustering_labels)
print(silhouette_avg)

ch_score = calinski_harabasz_score(dataset, clustering_labels)
print(ch_score)

db_score = davies_bouldin_score(dataset, clustering_labels)
print(db_score)